# 11.04 Claude Memory Middleware

Claude 네이티브 `memory_20250818` 도구를 붙이는 두 미들웨어 —
`StateClaudeMemoryMiddleware` 와 `FilesystemClaudeMemoryMiddleware` — 로 **장기 기억**을 구현한다.

Claude가 스스로 `/memories/*` 경로에 메모를 쓰고, 이후 대화에서 필요할 때 다시 읽어 참고한다.

## 학습 목표

- Claude 메모리 도구의 **`/memories/` 경로 계약** 을 이해한다
- State vs Filesystem 변형의 **지속성 범위** 차이를 구분한다
- 주입되는 `system_prompt` 가 모델에게 어떻게 "먼저 메모리를 확인하라" 를 지시하는지 안다
- Deep Agents 의 `StoreBackend` 와 어떻게 역할이 겹치고/다른지 안다

## 언제 쓰나

- 같은 사용자와 **여러 세션에 걸쳐** 취향/사실을 기억해야 하는 챗봇
- 에이전트가 장시간 작업 중 **중간 결과를 본인이 쓰고 본인이 다시 읽어야** 하는 경우
- RAG와 달리 **모델이 직접 메모 내용을 관리**(쓰고 지우고 수정)하도록 맡기고 싶을 때

## 11.04.1 환경 설정

필요 패키지: `langchain`, `langchain-anthropic`. `.env`에 `ANTHROPIC_API_KEY`.

In [ ]:
# !pip install -U langchain langchain-anthropic

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
model.invoke("ping").content[:50]

## 11.04.2 State 변형 — 스레드 내 지속

`StateClaudeMemoryMiddleware` 는 메모 내용을 LangGraph 상태(`memory_files`)에 넣는다.
`MemorySaver` / `PostgresCheckpointer` 등 체크포인터를 쓰면 **같은 thread_id 에서 재개할 때마다** 자동으로 복원된다.

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `allowed_path_prefixes` | `["/memories"]` | 메모 저장 허용 경로. 보통 그대로 둔다 |
| `system_prompt` | Anthropic 기본 프롬프트 | 매 턴 시작 전 '/memories 를 먼저 확인하라' 를 지시 |

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from langchain_anthropic.middleware import StateClaudeMemoryMiddleware

agent_mem = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[],
    middleware=[StateClaudeMemoryMiddleware()],
    checkpointer=MemorySaver(),
)

cfg = {"configurable": {"thread_id": "user-42"}}

# 1턴: 모델이 /memories 에 사용자 정보를 적도록 유도
r1 = agent_mem.invoke({
    "messages": [{
        "role": "user",
        "content": "내 이름은 지민이고 파이썬 백엔드 개발자야. 앞으로 대화에서 참고할 수 있게 이 사실을 메모리에 저장해줘.",
    }]
}, config=cfg)
print("--- 1턴 응답 ---")
print(r1["messages"][-1].content[:300])
print("메모리 파일:", list(r1.get("memory_files", {}).keys()))

## 11.04.3 같은 thread_id 로 재호출 → 메모 재사용

새 질문에서 모델이 **이름을 묻지 않고도** 이전 메모를 읽어 맞춰 답한다.
동일 `thread_id` 로 호출하면 체크포인터가 상태를 복원해 준다.

In [ ]:
r2 = agent_mem.invoke({
    "messages": [{
        "role": "user",
        "content": "내 주 언어를 기준으로 다음 주 학습 계획을 짜줘.",
    }]
}, config=cfg)
print("--- 2턴 응답 ---")
print(r2["messages"][-1].content[:400])
# 메모리에 남은 내용 점검
for path, content in r2.get("memory_files", {}).items():
    print(f"--- {path} ---"); print(content)

## 11.04.4 Filesystem 변형 — 프로세스/세션 경계를 넘어서 지속

`FilesystemClaudeMemoryMiddleware` 는 메모를 **실제 디스크 디렉터리**에 남긴다.
프로세스가 종료되거나 체크포인터 저장소가 달라도 디스크에 파일이 남아 있으면 그대로 다시 읽을 수 있다.

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `root_path` | (필수) | 메모리를 저장할 실제 디렉터리 |
| `allowed_prefixes` | `["/memories"]` | 메모 작성을 허용할 가상 경로 |
| `max_file_size_mb` | `10` | 한 메모 파일 최대 크기 |
| `system_prompt` | 기본 제공 | 메모리 활용 지시 프롬프트 |

In [ ]:
import tempfile, pathlib
from langchain_anthropic.middleware import FilesystemClaudeMemoryMiddleware

mem_root = pathlib.Path(tempfile.mkdtemp(prefix="claude-mem-"))

agent_mem_fs = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024),
    tools=[],
    middleware=[
        FilesystemClaudeMemoryMiddleware(
            root_path=str(mem_root),
            max_file_size_mb=1,
        ),
    ],
)

r = agent_mem_fs.invoke({
    "messages": [{
        "role": "user",
        "content": "다음 사실을 메모리에 저장해줘: 회의는 매주 화요일 오후 3시.",
    }]
})
print(r["messages"][-1].content[:300])
print("--- 디스크에 남은 파일들 ---")
for p in sorted((mem_root).rglob("*")):
    if p.is_file():
        print(p.relative_to(mem_root), "=>", p.read_text()[:120])

## 11.04.5 Deep Agents `StoreBackend` 와의 관계

Deep Agents 0.5는 영구 메모리를 위한 `StoreBackend` 를 별도로 제공한다. 둘은 **역할이 유사하지만 범위가 다르다**.

| 축 | Claude Memory Middleware | Deep Agents `StoreBackend` |
|----|-------------------------|----------------------------|
| 제공자 | Anthropic 네이티브 도구 | LangGraph Store API 래퍼 |
| 지원 모델 | Claude 전용 | 모든 모델 |
| 저장 구조 | `/memories/*` 파일 | `(namespace, key) -> dict` |
| 검색 방식 | 파일 목록 + 내용 읽기 | 임베딩 벡터 검색 지원 |
| 적합한 용도 | Claude가 자유롭게 메모를 관리 | 구조화된 프로필·사실 저장 |

**조합 팁**: 둘 다 쓸 수 있다. Claude가 단기 작업 노트를 `/memories` 에 남기고,
장기 프로필은 Deep Agents `StoreBackend` 에 넣어 다른 프로바이더 에이전트와 공유한다.

## 체크리스트

- [ ] State 변형 + 체크포인터로 **thread_id 단위 지속성**을 확인했는가
- [ ] Filesystem 변형의 `root_path` 를 별도 볼륨/디렉터리로 격리했는가
- [ ] 메모가 커지지 않도록 `max_file_size_mb` 를 적정히 설정했는가
- [ ] Deep Agents `StoreBackend` 와 **역할 분담** 을 정의했는가

## 다음 노트북

- `05_anthropic_file_search.ipynb` — 상태 내 파일에 대한 grep/glob 검색

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/anthropic
- Anthropic memory tool: https://docs.anthropic.com/claude/docs/tool-use-examples#memory-tool